# 04 · RAW → CLEAN — escala e cores (dbt/DuckDB)

Mede o **full-refresh** RAW→CLEAN em **1M / 10M / 100M** (eixo escala) e variando **threads**
(eixo cores). `dbt run` ≡ DuckDB `CREATE OR REPLACE TABLE ... AS <select>` — então medimos o SQL
que o dbt compilaria, direto no DuckDB.

**RAW:** `.data/dltcx_100m.duckdb` → `customers_w100m` (100M × 20 col, do folder 01).

> 📖 **Approach, interpretação e a decisão estão no [RESULTADOS.md](RESULTADOS.md).** Este notebook
> é só o experimento executável.

In [ ]:
# --- setup ---
import sys; sys.path.insert(0, "/Users/allanfraga/Repos/strattum/experimentacoes")
import exp, duckdb, os, json, subprocess

RAW_DB  = f"{exp.DATA}/dltcx_100m.duckdb"          # RAW: postgres.customers_w100m (100M x 20col)
TEST    = f"{exp.DATA}/clean_test"                  # tudo deste notebook mora aqui
DIMS_DB = f"{TEST}/dims.duckdb"                     # dim_partner (criada abaixo)
OUT_DB  = f"{TEST}/clean_out.duckdb"                # destino CLEAN (sobrescrito a cada run)
SPILL   = f"{TEST}/spill"                           # pra onde o DuckDB derrama quando aperta a RAM
os.makedirs(SPILL, exist_ok=True)

print("RAW  :", f"{duckdb.connect(RAW_DB, read_only=True).sql('SELECT count(*) FROM postgres.customers_w100m').fetchone()[0]:,}", "linhas x 20 col")
print("CPUs :", os.cpu_count(), "| RAM ~8 GB (memory_limit default do DuckDB ≈ 80% = 6.4 GB)")

### Dimensão fake (1.000 linhas)

`dim_partner` pra o `LEFT JOIN` de enriquecimento — broadcast (cabe na RAM, join barato). Cada
cliente recebe um partner via `1 + (abs(hash(id)) % 1000)`.

In [ ]:
# cria a dimensão fake de 1.000 linhas (idempotente)
d = duckdb.connect(DIMS_DB)
d.execute("""CREATE OR REPLACE TABLE dim_partner AS
  SELECT i + 1                                       AS partner_id,
         'Partner ' || (i + 1)                       AS partner_name,
         (['platinum','gold','silver','bronze'])[1+(i%4)] AS partner_tier,
         round(0.01 + (i % 20) * 0.005, 3)           AS commission_rate
  FROM range(1000) t(i)""")
print("dim_partner:", f"{d.sql('SELECT count(*) FROM dim_partner').fetchone()[0]:,}", "linhas")
display(d.sql("SELECT * FROM dim_partner ORDER BY partner_id LIMIT 3").df())
d.close()

### O modelo CLEAN

Várias transformações numa tacada — normalização (`LOWER/TRIM`, `external_id`, UTC, `CASE`),
enriquecimento (`LEFT JOIN dim_partner`) e uma **window** (`rank()`, o item caro). Definido **uma
vez** em `CLEAN_SQL` e reusado no smoke e no benchmark.

> Bate com o padrão real da clean (`materialized=table`, `LOWER/TRIM`, `LEFT JOIN`); a `rank()` é
> stress proposital — nenhum dos 42 transforms reais usa window. Detalhe no README.

In [ ]:
# O corpo do modelo CLEAN. {raw_from} = a fonte (com LIMIT por escala, ou os 100M inteiros).
CLEAN_SQL = """
CREATE OR REPLACE TABLE clean.clean_customers AS
SELECT
    c.id                                     AS external_id,        -- regra clean: external_id
    lower(trim(c.email))                     AS email,              -- LOWER/TRIM email
    split_part(lower(trim(c.email)), '@', 2) AS email_domain,       -- transform de string
    trim(c.name)                             AS name,
    c.phone, c.city, c.country, c.region, c.segment,
    c.age, c.score, c.balance, c.credit_limit, c.is_active,
    c.created_at AT TIME ZONE 'UTC'          AS created_at_utc,     -- normalização UTC
    date_diff('day', c.created_at, TIMESTAMP '2026-06-27') AS account_age_days,  -- derivado de data
    CASE WHEN c.balance >= 1000 THEN 'high'
         WHEN c.balance >= 100  THEN 'mid'
         ELSE 'low' END                      AS balance_tier,       -- segmentação CASE
    p.partner_name, p.partner_tier, p.commission_rate,              -- enrich via dim 1.000 linhas
    round(c.balance * p.commission_rate, 2)  AS commission_value,   -- derivado do join
    rank() OVER (PARTITION BY c.segment
                 ORDER BY c.balance DESC)     AS balance_rank_in_segment  -- WINDOW (o caro)
FROM {raw_from} c
LEFT JOIN dims.dim_partner p ON (1 + (abs(hash(c.id)) % 1000)) = p.partner_id  -- broadcast
WHERE c.id IS NOT NULL
"""

def open_out(memory=None, threads=8):
    """Conexão de saída: anexa RAW + dims (read-only), schema clean, spill em disco."""
    if os.path.exists(OUT_DB): os.remove(OUT_DB)
    c = duckdb.connect(OUT_DB)
    c.execute(f"ATTACH '{RAW_DB}'  AS raw  (READ_ONLY)")
    c.execute(f"ATTACH '{DIMS_DB}' AS dims (READ_ONLY)")
    c.execute("CREATE SCHEMA IF NOT EXISTS clean")
    c.execute("SET enable_progress_bar=false")
    c.execute(f"SET threads={threads}")
    c.execute(f"SET temp_directory='{SPILL}'")
    if memory:
        c.execute(f"SET memory_limit='{memory}'")
    return c

# smoke test: roda o modelo numa AMOSTRA de 1M, valida o schema e espia o resultado
c = open_out()
c.execute(CLEAN_SQL.format(raw_from="(SELECT * FROM raw.postgres.customers_w100m LIMIT 1000000)"))
out = c.sql("SELECT * FROM clean.clean_customers")
print("smoke 1M · linhas:", f"{c.sql('SELECT count(*) FROM clean.clean_customers').fetchone()[0]:,}",
      "· colunas:", len(out.columns))
display(c.sql("""SELECT external_id, email, email_domain, balance_tier,
                        partner_name, commission_value, balance_rank_in_segment
                 FROM clean.clean_customers ORDER BY external_id LIMIT 5""").df())
c.close()

### Benchmark — escala + cores

Cada config roda em **subprocesso novo** (RSS é pico cumulativo do processo, tem que isolar).
**Eixo A:** 1M/10M/100M a 8 threads. **Eixo B:** 10M a 1/2/4/8 threads.

> ⏱️ O 100M leva ~13 min (a `rank()` ordena 100M e derrama pra disco). Garanta ~25 GB livres.

In [ ]:
import pandas as pd

# Runner em PROCESSO SEPARADO (RSS limpo). argv: limit threads. memory = default.
# CLEAN_SQL é injetado como literal (replace de __SQL__, pra não brigar com as chaves {}).
RUNNER_TEMPLATE = '''
import duckdb, time, resource, os, json, sys
limit = int(sys.argv[1]); threads = int(sys.argv[2])
RAW_DB, DIMS_DB, OUT_DB, SPILL = sys.argv[3], sys.argv[4], sys.argv[5], sys.argv[6]
SQL = __SQL__
raw = "raw.postgres.customers_w100m" if limit == 0 else \\
      "(SELECT * FROM raw.postgres.customers_w100m LIMIT " + str(limit) + ")"
def rss():
    v = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return v / 1e6 if v > 1e6 else v / 1e3
if os.path.exists(OUT_DB):
    os.remove(OUT_DB)
con = duckdb.connect(OUT_DB)
con.execute("ATTACH '" + RAW_DB + "' AS raw (READ_ONLY)")
con.execute("ATTACH '" + DIMS_DB + "' AS dims (READ_ONLY)")
con.execute("CREATE SCHEMA IF NOT EXISTS clean")
con.execute("SET enable_progress_bar=false")
con.execute("SET threads=" + str(threads))
con.execute("SET temp_directory='" + SPILL + "'")
t0 = time.perf_counter()
con.execute(SQL.format(raw_from=raw))            # == dbt run --full-refresh
el = time.perf_counter() - t0
rows = con.execute("SELECT count(*) FROM clean.clean_customers").fetchone()[0]
con.close()
print(json.dumps({"limit": limit, "threads": threads, "rows": rows,
                  "seconds": round(el, 2), "rss_mb": round(rss())}))
'''
RUNNER = RUNNER_TEMPLATE.replace("__SQL__", repr(CLEAN_SQL))

def bench(limit, threads):
    r = subprocess.run([sys.executable, "-c", RUNNER, str(limit), str(threads),
                        RAW_DB, DIMS_DB, OUT_DB, SPILL], capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  FALHOU limit={limit}/{threads}t:\n{r.stderr[-1200:]}"); return None
    return json.loads(r.stdout.strip().splitlines()[-1])

# Eixo A — escala (threads=8)
escala = []
for lim in (1_000_000, 10_000_000, 0):
    print(f"escala: {lim or 100_000_000:,} ...", flush=True)
    res = bench(lim, 8)
    if res: escala.append(res); print("  ->", res, flush=True)

# Eixo B — cores (10M, 1/2/4/8 threads)
cores = []
for th in (1, 2, 4, 8):
    print(f"cores: 10M @ {th}t ...", flush=True)
    res = bench(10_000_000, th)
    if res: cores.append(res); print("  ->", res, flush=True)

df_escala = pd.DataFrame(escala).assign(rows=lambda d: d["rows"].map("{:,}".format))
df_cores = pd.DataFrame(cores)
print("\n== ESCALA (threads=8) =="); display(df_escala[["rows", "threads", "seconds", "rss_mb"]])
print("== CORES (10M) ==");          display(df_cores[["threads", "seconds", "rss_mb"]])

## Resultados (medidos: 8 cores, 8 GB RAM, SSD)

**Escala (8 threads):**

| Linhas | tempo | RSS |
|---|---|---|
| 1M | 2,9 s | 857 MB |
| 10M | 23,3 s | 2,2 GB |
| 100M | 812 s (~13,5 min) | 3,2 GB |

→ ~linear até caber na RAM; **super-linear no 100M** (a `rank()` ordena 100M → spill domina).
RAM **plana** (3,2 GB no 100M, teto 6,4 GB) = **out-of-core, não estoura**.

**Cores (10M, a frio):**

| threads | tempo | speedup |
|---|---|---|
| 1 | 93,9 s | 1,00× |
| 2 | 48,8 s | 1,93× |
| 4 | 26,8 s | 3,51× |
| 8 | 22,4 s | 4,19× |

→ quase linear até 4 threads (**3,5×**), depois satura (**4→8 = 1,2×**). `threads:4` é sweet spot.

> 📖 **Interpretação completa, comparação com a prod e a DECISÃO no [RESULTADOS.md](RESULTADOS.md).**
> TL;DR: dbt/DuckDB escala e está ótimo — manter; só conferir `threads` = cores do box.